<a href="https://colab.research.google.com/github/hsin0606/MachineLearning/blob/main/%E6%9C%9F%E6%9C%AB%E4%BD%9C%E6%A5%AD/0704_Colab_LINE_Bot_with_GEMINI_Tooluse_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install Flask pyngrok line-bot-sdk requests --quiet
!pip install google-genai --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.5/819.5 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.6/165.6 kB 8.0 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

ngrok_authtoken = userdata.get('NGROK_AUTHTOKEN')
line_channel_access_token = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
line_channel_secret = userdata.get('LINE_CHANNEL_SECRET')
gemini_api_key = userdata.get('GEMINI_API_KEY')
port = 5051


In [ ]:
import os
from pyngrok import ngrok

In [ ]:
ngrok.kill()

In [ ]:
import requests

ngrok.set_auth_token(ngrok_authtoken)
tunnel = ngrok.connect(5051, name="linebot_tunnel")
webhook_url = tunnel.public_url

print(f"Ngrok URL: {webhook_url}")

# 自動更新 LINE Webhook URL
def update_line_webhook(webhook_url):
    """使用 LINE Messaging API 更新 Webhook URL"""
    url = "https://api.line.me/v2/bot/channel/webhook/endpoint"
    headers = {
        "Authorization": f"Bearer {line_channel_access_token}",
        "Content-Type": "application/json"
    }
    data = {
        "endpoint": webhook_url
    }

    response = requests.put(url, headers=headers, json=data)

    if response.status_code == 200:
        print(f"✅ LINE Webhook URL 已自動更新為：{webhook_url}")
        return True
    else:
        print(f"❌ 更新失敗：{response.status_code} - {response.text}")
        return False

# 執行更新
update_line_webhook(webhook_url)

Ngrok URL: https://colonize-lego-runt.ngrok-free.dev
✅ LINE Webhook URL 已自動更新為：https://colonize-lego-runt.ngrok-free.dev


True

In [ ]:
from google import genai
from google.genai.types import Tool, GenerateContentConfig, GoogleSearch

# === 初始化 Google Gemini ===
client = genai.Client(api_key=gemini_api_key)

google_search_tool = Tool(
   google_search=GoogleSearch()    #校長這邊用了google search
)

chat = client.chats.create(
    model="gemini-2.5-flash",
    config=GenerateContentConfig(
        system_instruction="你是一個中文的AI助手，請用繁體中文回答",
        tools=[google_search_tool],
        response_modalities=["TEXT"],
    )
)

In [ ]:
def stateful_query(payload):
    response = chat.send_message(message=payload)
    return response.text

In [ ]:
result = stateful_query("簡介明新科技大學")
print(result)

明新科技大學（Minghsin University of Science and Technology），簡稱明新科大，是一所位於臺灣新竹縣新豐鄉的私立科技大學。

**發展沿革：**
明新科大的歷史可追溯至1966年3月1日創立的「明新工業專科學校」，初期設有機械、土木、工業管理三科。 1993年，更名為「明新工商專科學校」。 隨著教育體制的發展，1997年7月改制為「明新技術學院」並附設專科部。 2002年8月，獲教育部核准優先升格為科技大學，正式定名為「明新科技大學」。 2018年12月，學校更名為「明新學校財團法人明新科技大學」。

**學術組織：**
學校目前設有六個學院，包括半導體學院、工程學院、管理學院、民生學院、人文與設計學院、共同教育學院。 涵蓋了20個學系、2個學位學程（含1個博士學位學程）以及11個碩士班。 其中，半導體學院的成立與半導體科技博士學位學程的核准，顯示學校積極投入前瞻科技領域。

**辦學理念與特色：**
明新科技大學以「在明明德，在新民，在止於至善」為校名精義，並以「堅毅、求新、創造」為校訓。 其願景為「深耕在地、放眼國際」，目標是培養「具實務經驗與人文素養之專業人才」。

學校地處新竹縣新豐鄉，鄰近新竹工業區與科學園區，地理位置優越，交通便利。 憑藉地利之便，明新科大與產業界建立了多項產學合作計畫，為區域產業培養了大量人才，被譽為新竹縣人才供應的第一品牌。 學校也積極推動「MUST」四大育才特色：多元學習、全球視野、永續經營與技術創新，旨在培養學生跨域學習能力，成為具備實務經驗和人文素養的專業人才，特別鎖定半導體、AI、元宇宙、風電綠能等前瞻產業。 明新科大在企業最愛公私立技職科大調查中表現亮眼，並被定位為「一流產業大學」。

學校亦積極推動國際化，吸引來自亞洲、美洲、歐洲等地的國際學生，全校學生總數超過一萬人，其中外籍生約一千人。


In [ ]:
result2 = stateful_query("校長是誰？")
print(result2)

None


In [ ]:
from flask import Flask, request, abort

from linebot.v3 import (
    WebhookHandler
)
from linebot.v3.exceptions import (
    InvalidSignatureError
)
from linebot.v3.messaging import (
    Configuration,
    ApiClient,
    MessagingApi,
    ReplyMessageRequest,
    TextMessage,
)
from linebot.v3.webhooks import (
    MessageEvent,
    TextMessageContent,
)

app = Flask(__name__)

configuration = Configuration(access_token=line_channel_access_token)
handler = WebhookHandler(line_channel_secret)


@app.route("/", methods=['POST'])
def callback():
    # get X-Line-Signature header value
    signature = request.headers['X-Line-Signature']

    # get request body as text
    body = request.get_data(as_text=True)
    print("BODY: ", body)
    app.logger.info("Request body: " + body)

    # handle webhook body
    try:
        handler.handle(body, signature)
    except InvalidSignatureError:
        app.logger.info("Invalid signature. Please check your channel access token/channel secret.")
        abort(400)

    return 'OK'


@handler.add(MessageEvent, message=TextMessageContent)
def handle_message(event):
    text = event.message.text
    with ApiClient(configuration) as api_client:
        line_bot_api = MessagingApi(api_client)
        if text.startswith('AI '):
            prompt = text[3:]
            reply_text = stateful_query(prompt)
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=reply_text)]
                )
            )

        else:
            line_bot_api.reply_message_with_http_info(
                ReplyMessageRequest(
                    reply_token=event.reply_token,
                    messages=[TextMessage(text=event.message.text),
                        TextMessage(text=event.message.text)]
                )
            )

if __name__ == "__main__":
    app.run(port=port)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5051
INFO:werkzeug:Press CTRL+C to quit
